# Side experiment: FunctionGemma 270M as the question → JSON plan translator

A separate try, apart from the main runs: the same v10 data and the same `train.py`, but Google's 270M function-calling
model instead of Qwen3-0.6B. It is half the size on the phone (159 MB instead of 341 MB at INT4) and about twice as fast.
The question is whether it is as accurate.

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Keep this tab open; about 1.5–2.5 h.
2. Nothing goes to Google Drive. At the end the browser downloads `fg270-v10qat-summary.txt`, the plans for the
   test and probe sets (`*.predictions.jsonl`) and the model (`fg270-v10qat.zip`, about 0.5 GB). Allow multiple downloads if asked.

In [ ]:
# Settings
MODEL = "unsloth/functiongemma-270m-it"   # an open copy of google/functiongemma-270m-it (that one needs a Hugging Face login)
RUN_NAME = "fg270-v10qat"
BATCH = 8
EPOCHS = 2
LR = 2e-4
QAT = True          # train against the phone's INT4 rounding
FP16 = False        # Gemma models can overflow in float16; the T4 has no bfloat16, so train in float32

import os
OUT = f"/content/runs/{RUN_NAME}"
os.makedirs(os.path.dirname(OUT), exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# The library versions train.py was tested with
!pip install -q transformers==5.22.0 trl==1.17.0 peft==0.21.2 datasets==5.1.0 accelerate==1.15.0
!pip uninstall -q -y torchao

In [ ]:
# Scripts and data (the main branch: v10 data); this run's only change to train.py is float32 instead of float16
!rm -rf /content/repo && git clone -q --depth 1 https://github.com/fatihsokmen/plan-translator-colab /content/repo
!tar xzf /content/repo/data.tar.gz -C /content/repo && wc -l /content/repo/data/*/*.jsonl
if not FP16:
    !sed -i 's/fp16=torch.cuda.is_available() and not bf16,/fp16=False,/' /content/repo/train.py
    !grep -n "fp16=" /content/repo/train.py

In [ ]:
# Training. Prints the dev exact-plan rate at the end.
!cd /content/repo && python train.py --model {MODEL} --data data --out {OUT} --bs {BATCH} --save_steps 1000 --epochs {EPOCHS} --lr {LR} {"--qat" if QAT else ""} 2>&1 | tee -a {OUT}.log
assert os.path.exists(f"{OUT}/final"), "training did not finish: the error is in the output above"

In [ ]:
# The dev score, then plans for the test and probe sets (the app scores these; blind_v4 by counts only)
from google.colab import files
log = [l.strip() for l in open(f"{OUT}.log")]
summary = "\n".join([RUN_NAME] + [l for l in log if l.startswith(("dev: exact plan", "dev int4"))][-2:]) + "\n"
open(f"/content/{RUN_NAME}-summary.txt", "w").write(summary)
print(summary)
files.download(f"/content/{RUN_NAME}-summary.txt")
for name, prompts in [("real_like_v2", "data/test/real_like_v2.prompts.jsonl"), ("independent_v3", "data/test/independent_v3.prompts.jsonl"),
                      ("probe_combos_v1", "data/probe/combos_v1.prompts.jsonl"), ("independent_v3_typos", "data/probe/independent_v3_typos.prompts.jsonl"),
                      ("blind_v4", "data/test/blind_v4.prompts.jsonl"), ("shapes_v7", "data/probe/shapes_v7.prompts.jsonl"),
                      ("weak_v9", "data/probe/weak_v9.prompts.jsonl"), ("phone_v1", "data/probe/phone_v1.prompts.jsonl")]:
    !cd /content/repo && python predict.py --model {OUT}/final --prompts {prompts} --out /content/{RUN_NAME}.{name}.predictions.jsonl
    files.download(f"/content/{RUN_NAME}.{name}.predictions.jsonl")

In [ ]:
# The model itself, as float16 (about 0.5 GB); Colab deletes it when the session ends
import torch, shutil
from transformers import AutoModelForCausalLM, AutoTokenizer
m = AutoModelForCausalLM.from_pretrained(f"{OUT}/final", dtype=torch.float16)
m.save_pretrained("/content/model_fp16")
AutoTokenizer.from_pretrained(f"{OUT}/final").save_pretrained("/content/model_fp16")
shutil.make_archive(f"/content/{RUN_NAME}", "zip", "/content/model_fp16")
files.download(f"/content/{RUN_NAME}.zip")